# Hands-on Task 3: Joint Representation Learning and Clustering

1.Use the pretrained AE encoder from Task 1 and Initialize cluster centers by applying K-means to the latent representations.

In [7]:
# 1. 导入库和加载 MNIST
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.cluster import KMeans

torch.manual_seed(47)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 16
n_clusters = 10
batch_size = 256
epochs = 10

data_root = "/Users/kenny/Research/notebook/research-preparation-federated-clustering/data"

train_dataset = datasets.MNIST(
    root=data_root,
    train=True,
    download=False,
    transform=transforms.ToTensor(),
)

# 训练时打乱样本
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

# 提取特征时保持原始样本顺序
feature_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
)


# 2. 构建 AE，与 Task 1 的结构一致
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),                  # [B, 1, 28, 28] → [B, 784]
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),      # [B, 16]
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),   # [B, 1, 28, 28]
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed


model = Autoencoder(latent_dim).to(device)


# 3. 预训练 AE：让重建图片尽量接近原图
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for images, _ in train_loader:
        images = images.to(device)

        optimizer.zero_grad()

        reconstructed = model(images)
        loss = criterion(reconstructed, images)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    average_loss = train_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")


# 4. 使用预训练后的编码器提取潜在特征
model.eval()
latent_batches = []

with torch.no_grad():
    for images, _ in feature_loader:
        z = model.encoder(images.to(device))  # [B, 16]
        latent_batches.append(z.cpu())

Z_init = torch.cat(latent_batches, dim=0).numpy()  # [60000, 16]


# 5. 对潜在特征运行 K-means，得到初始聚类中心
kmeans_init = KMeans(
    n_clusters=n_clusters,
    n_init=10,
    random_state=47,
)

initial_labels = kmeans_init.fit_predict(Z_init)

cluster_centers = torch.tensor(
    kmeans_init.cluster_centers_,
    dtype=torch.float32,
    device=device,
)

Epoch [1/10], Loss: 0.2287
Epoch [2/10], Loss: 0.1960
Epoch [3/10], Loss: 0.1103
Epoch [4/10], Loss: 0.0786
Epoch [5/10], Loss: 0.0735
Epoch [6/10], Loss: 0.0720
Epoch [7/10], Loss: 0.0713
Epoch [8/10], Loss: 0.0708
Epoch [9/10], Loss: 0.0704
Epoch [10/10], Loss: 0.0700


In [8]:
from copy import deepcopy
import torch
from torch import nn
from torch.utils.data import DataLoader

# 保存原始 AE 和初始聚类结果，供 Option A/B 独立使用
baseline_model = deepcopy(model)
baseline_labels = initial_labels.copy()

baseline_centers = torch.tensor(
    kmeans_init.cluster_centers_.copy(),
    dtype=next(model.parameters()).dtype,
    device=next(model.parameters()).device,
)

2.1 option a：交替更新
用当前聚类结果指导编码器训练。

用更新后的编码器重新提取特征。

重新运行 K-means，更新簇标签和聚类中心。

In [9]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans


# 1. 给每张图片附上索引
# 即使训练时打乱图片，也能找到它对应的簇标签
class IndexedDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, _ = self.dataset[index]  # 忽略真实数字标签
        return image, index


# 每次运行 Option A 都从保存的原始 AE 开始
model = deepcopy(baseline_model)
device = next(model.parameters()).device

refine_loader = DataLoader(
    IndexedDataset(train_dataset),
    batch_size=256,
    shuffle=True,
)

# 提取特征时保持图片顺序
ordered_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
)


# 2. 用当前编码器提取全部图片的潜在特征
def extract_latent():
    model.eval()
    batches = []

    with torch.no_grad():
        for images, _ in ordered_loader:
            z = model.encoder(images.to(device))
            batches.append(z.cpu())

    return torch.cat(batches, dim=0).numpy()


# 3. 从上一步的初始聚类结果开始
refined_labels = baseline_labels.copy()
refined_centers = baseline_centers.detach().clone().to(device)
n_clusters = refined_centers.shape[0]

refine_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse = nn.MSELoss()

rounds = 5              # 交替进行 5 轮
epochs_per_round = 1   # 每轮先训练 AE 1 个 epoch
lambda_cluster = 0.001  # 聚类损失的权重


# 4. 交替更新 AE 和 K-means
for round_idx in range(rounds):

    # A1：固定本轮的簇标签和中心，训练 AE
    labels_tensor = torch.as_tensor(
        refined_labels,
        dtype=torch.long,
        device=device,
    )

    for epoch in range(epochs_per_round):
        model.train()
        reconstruction_sum = 0.0
        clustering_sum = 0.0

        for images, indices in refine_loader:
            images = images.to(device)
            indices = indices.to(device)

            # 根据图片索引，取出该图片所属簇的中心
            target_centers = refined_centers[labels_tensor[indices]]

            refine_optimizer.zero_grad()

            z = model.encoder(images)         # [B, 16]
            reconstructed = model.decoder(z)  # [B, 1, 28, 28]

            # 保留重建能力，并让特征靠近所属中心
            reconstruction_loss = mse(reconstructed, images)
            clustering_loss = mse(z, target_centers)

            loss = (
                reconstruction_loss
                + lambda_cluster * clustering_loss
            )

            loss.backward()
            refine_optimizer.step()

            reconstruction_sum += (
                reconstruction_loss.item() * images.size(0)
            )
            clustering_sum += (
                clustering_loss.item() * images.size(0)
            )

        print(
            f"Round {round_idx + 1}/{rounds}, "
            f"Epoch {epoch + 1}/{epochs_per_round}, "
            f"Reconstruction: "
            f"{reconstruction_sum / len(train_dataset):.4f}, "
            f"Clustering: "
            f"{clustering_sum / len(train_dataset):.4f}"
        )

    # A2：用更新后的编码器重新提取特征
    Z_refined = extract_latent()

    # 重新运行 K-means，更新簇标签和中心
    kmeans_refined = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=47,
    )

    refined_labels = kmeans_refined.fit_predict(Z_refined)

    refined_centers = torch.as_tensor(
        kmeans_refined.cluster_centers_,
        dtype=next(model.parameters()).dtype,
        device=device,
    )


# 5. 保存最终结果
cluster_centers = refined_centers
cluster_labels = refined_labels

Round 1/5, Epoch 1/1, Reconstruction: 0.0670, Clustering: 0.6578
Round 2/5, Epoch 1/1, Reconstruction: 0.0555, Clustering: 1.0621
Round 3/5, Epoch 1/1, Reconstruction: 0.0444, Clustering: 1.2662
Round 4/5, Epoch 1/1, Reconstruction: 0.0401, Clustering: 1.2513
Round 5/5, Epoch 1/1, Reconstruction: 0.0361, Clustering: 1.3534


In [10]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

# 真实数字标签只用于最终评价
# 与两组聚类结果保持相同的图片顺序
y_true = train_dataset.targets.cpu().numpy()

assert initial_labels.shape == refined_labels.shape == y_true.shape

# 基线：AE + fixed K-means
baseline_ari = adjusted_rand_score(y_true, initial_labels)
baseline_nmi = normalized_mutual_info_score(y_true, initial_labels)

# 改进后：Option A
refined_ari = adjusted_rand_score(y_true, refined_labels)
refined_nmi = normalized_mutual_info_score(y_true, refined_labels)

# 打印比较表
print(f"{'Method':<24} {'ARI':>10} {'NMI':>10}")

print(
    f"{'AE + fixed K-means':<24} "
    f"{baseline_ari:>10.4f} {baseline_nmi:>10.4f}"
)

print(
    f"{'Refined model (Option A)':<24} "
    f"{refined_ari:>10.4f} {refined_nmi:>10.4f}"
)

print(
    f"{'Change':<24} "
    f"{refined_ari - baseline_ari:>+10.4f} "
    f"{refined_nmi - baseline_nmi:>+10.4f}"
)

Method                          ARI        NMI
AE + fixed K-means           0.0209     0.0510
Refined model (Option A)     0.2741     0.3858
Change                      +0.2532    +0.3348


2.2 option b：给原来的损失增加一项聚类损失

In [11]:
# 从原始预训练 AE 创建独立模型
model_b = deepcopy(baseline_model)
device_b = next(model_b.parameters()).device

# 聚类中心设为可训练参数
centers_b = nn.Parameter(
    baseline_centers.detach().clone().to(device_b)
)

train_loader_b = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
)

# 同时优化 AE 参数和聚类中心
optimizer_b = torch.optim.Adam(
    list(model_b.parameters()) + [centers_b],
    lr=1e-4,
)

mse_b = nn.MSELoss()
lambda_cluster_b = 0.001
epochs_b = 5

for epoch in range(epochs_b):
    model_b.train()
    reconstruction_sum = 0.0
    clustering_sum = 0.0

    for images, _ in train_loader_b:
        images = images.to(device_b)
        optimizer_b.zero_grad()

        z = model_b.encoder(images)           # [B, 16]
        reconstructed = model_b.decoder(z)   # [B, 1, 28, 28]

        # 1. 重建损失
        reconstruction_loss = mse_b(reconstructed, images)

        # 2. 每张图片到每个中心的平方距离
        # 按特征维度取平均，与 MSE 的尺度一致
        distances = (
            z[:, None, :] - centers_b[None, :, :]
        ).pow(2).mean(dim=2)                  # [B, 10]

        # 每张图片取最近中心的距离，再对批次取平均
        clustering_loss = distances.min(dim=1).values.mean()

        # 3. 总损失
        loss = (
            reconstruction_loss
            + lambda_cluster_b * clustering_loss
        )

        loss.backward()
        optimizer_b.step()

        reconstruction_sum += (
            reconstruction_loss.item() * images.size(0)
        )
        clustering_sum += (
            clustering_loss.item() * images.size(0)
        )

    print(
        f"Option B, Epoch {epoch + 1}/{epochs_b}, "
        f"Reconstruction: "
        f"{reconstruction_sum / len(train_dataset):.4f}, "
        f"Clustering: "
        f"{clustering_sum / len(train_dataset):.4f}"
    )


# 按图片原始顺序，得到 Option B 的最终簇标签
eval_loader_b = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
)

model_b.eval()
label_batches_b = []

with torch.no_grad():
    for images, _ in eval_loader_b:
        z = model_b.encoder(images.to(device_b))

        # 分配到训练后最近的中心
        labels = torch.cdist(z, centers_b).argmin(dim=1)
        label_batches_b.append(labels.cpu())

labels_b = torch.cat(label_batches_b).numpy()

Option B, Epoch 1/5, Reconstruction: 0.0667, Clustering: 0.1389
Option B, Epoch 2/5, Reconstruction: 0.0573, Clustering: 0.9108
Option B, Epoch 3/5, Reconstruction: 0.0456, Clustering: 1.3422
Option B, Epoch 4/5, Reconstruction: 0.0407, Clustering: 1.2093
Option B, Epoch 5/5, Reconstruction: 0.0375, Clustering: 1.2140


In [12]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

# 真实数字标签只在最终评价时使用
y_true = train_dataset.targets.cpu().numpy()

assert baseline_labels.shape == labels_b.shape == y_true.shape

baseline_ari = adjusted_rand_score(y_true, baseline_labels)
baseline_nmi = normalized_mutual_info_score(y_true, baseline_labels)

b_ari = adjusted_rand_score(y_true, labels_b)
b_nmi = normalized_mutual_info_score(y_true, labels_b)

print(f"{'Method':<24} {'ARI':>10} {'NMI':>10}")

print(
    f"{'AE + fixed K-means':<24} "
    f"{baseline_ari:>10.4f} {baseline_nmi:>10.4f}"
)

print(
    f"{'Refined model (Option B)':<24} "
    f"{b_ari:>10.4f} {b_nmi:>10.4f}"
)

print(
    f"{'Change':<24} "
    f"{b_ari - baseline_ari:>+10.4f} "
    f"{b_nmi - baseline_nmi:>+10.4f}"
)

Method                          ARI        NMI
AE + fixed K-means           0.0209     0.0510
Refined model (Option B)     0.0386     0.0744
Change                      +0.0178    +0.0234


表示学习与聚类通过共享编码器参数相互影响。重建目标鼓励保留输入信息，聚类目标鼓励特征向中心聚集。两者可能相互促进，也可能产生不利影响。本实验中，refinement 后的 ARI 和 NMI 低于基线，说明当前聚类约束未改善类别区分能力，联合优化的损失下降并不保证聚类评价指标提高。